In [1]:
from exafs_shell_fit import *

Loading BokehJS ...

## 0. Ruuning feff Path

In [2]:
#%%script false
#1) Folder containing your FEFF run files
runner = FeffFitFramework(
    feff_folder="/scratch/ufondup/Ti_foil",  # Folder containing all FEFF calculation files
    feff_input="feff.inp",                                          # FEFF input file (geometry, potentials, etc.)
    use_feff8=True                                                   # Use FEFF version 8 (True) or older FEFF versions (False)
)

In [3]:
#%%script false
# 2) Run Feff to Calculate Path
runner.run_feff(verbose=True)

 : ======== running Feff module feff8l_rdinp ========
 : Feff8L (EXAFS)      release  0.1
 : Inconsistency in POTENTIAL card is detected for unique pot    1
 : Results might be meaningless.
 : Formula: Ti
 : SpaceGroup: P6_3/mmc
 : ======== running Feff module feff8l_pot ========
 : Calculating potentials ...
 : free atom potential and density for atom type    0
 : free atom potential and density for atom type    1
 : initial state energy
 : overlapped potential and density for unique potential    0
 : overlapped potential and density for unique potential    1
 : muffin tin radii and interstitial parameters
 : : ipot, Norman radius, Muffin tin radius, Overlap
 : 0  1.58586E+00  1.54199E+00  1.07112E+00
 : 1  1.60667E+00  1.55974E+00  1.07550E+00
 : : mu_old=   -12.240
 : Done with module 1: potentials.
 : ======== running Feff module feff8l_xsph ========
 : Calculating cross-section and phases...
 : absorption cross section
 : phase shifts for unique potential    0
 : phase shifts for 

## 1. Fitting 

In [ ]:
model = fit(
    datafile="/scratch/ufondup/Ti_foil/Ti_mu_Scan.dat",
    feff_dir="/scratch/ufondup/Ti_foil",
    preset="quick",
    kmin=2.0, kmax=10.0, kweight=2,
    rmin=1.5, rmax=7.0, rbkg=1.0,

    include_ms=True,
    ms_prune=True,
    # max_nfev=5000000,
    
    # Optional: solver cascade that plays nicely with covariance
    method='nelder',
    methods_try=('nelder', 'leastsq', 'brute'),

    plots=True,
    verbose=True,
)


[SNR autocut] kmax 10.00 → 9.50 (SNR threshold=2.0)
Identified SS shells: [(1, 2.905, 2, 12), (2, 4.108, 1, 6), (3, 4.641, 1, 2), (4, 5.067, 2, 18), (5, 5.492, 1, 12), (6, 5.872, 1, 6), (7, 6.537, 1, 12), (8, 6.884, 1, 12)]
[MS prune] kept 55 / 55 (dropped 0)
Prepared 65 FEFF paths (10 SS + 55 MS)
Identified SS shells: [(1, 2.905, 2, 12), (2, 4.108, 1, 6), (3, 4.641, 1, 2), (4, 5.067, 2, 18), (5, 5.492, 1, 12), (6, 5.872, 1, 6), (7, 6.537, 1, 12), (8, 6.884, 1, 12)]
[MS prune] kept 55 / 55 (dropped 0)
Prepared 65 FEFF paths (10 SS + 55 MS)

===== Per-shell refinement: shell 1 =====
 [shell 1] r-factor=0.011428338550366514, red-chi=1.143336965963895e-10, window=(2.655,3.155)

===== Per-shell refinement: shell 2 =====
 [shell 2] r-factor=0.7063870754124265, red-chi=1.4046706584689862e-09, window=(3.858,4.358)

===== Per-shell refinement: shell 3 =====
[auto-bounds] Expanded bounds for:
  - sig2_3: hit=min  bounds (0.001, 0.03) → (0.0005, 0.03) (count=1)
 [shell 3] r-factor=0.836336099210

In [ ]:
model.print_shell_summary()

In [ ]:
# Reports
print(model.report())

In [ ]:
# Saved Checkpoint
save_fit_params_json(model, "/gpfs/Scratch/ufondup/Ti_sample/Ti_fit_checkpoint.json", save_vary=True)

## 2. Start fitting from Save Params 

In [ ]:
# Build model with per-shell amplitudes enabled
model = FeffitAutoShellModel(
    datafile="/scratch/ufondup/Ti_foil/Ti_mu_Scan.dat",
    feff_dir="/scratch/ufondup/Ti_foil",
    # windows/transforms
    kmin=2.0, kmax=15.0, kweight=3,
    rmin=1.0, rmax=7.0, rbkg=1.0,
    include_ms=True,
    ms_prune=True,
    # max_nfev=5_000_000,
    # NEW: per-shell amplitudes Nscale_j
    per_shell_nscale=False,)

# Seed from JSON (values + vary flags as in the file)
seed_from_json(
    model,
    "/gpfs/Scratch/ufondup/Ti_sample/Ti_fit_checkpoint.json",
    apply_vary="from_json",
    show_after=True
)

# Run the fit (use your preferred method)
out= model.fit(
    verbose=True,
    # Optional: solver cascade that plays nicely with covariance
    method='leastsq',  #'brute'
    methods_try=(None, 'leastsq', None),
)

In [ ]:
model.plot_k()

In [ ]:
model.plot_r()

In [ ]:
# Reports
model.print_shell_summary()

In [ ]:
# Reports
print(model.report())


In [ ]:
# Saved Checkpoint
save_fit_params_json(model, "/gpfs/Scratch/ufondup/Ti_sample/Ti_fit_checkpoint1.json", save_vary=True)

## 3. Runing one iteration without changing the parameters 

In [ ]:

# 1) Build the model (do NOT run fit_v3 here, we want full control)
model = FeffitAutoShellModel(
    datafile="/scratch/ufondup/Ti_foil/Ti_mu_Scan.dat",
    feff_dir="/scratch/ufondup/Ti_foil",
    # set your usual windows; these just define transforms
    kmin=2.0, kmax=15.0, kweight=3,
    rmin=1.0, rmax=4.0, rbkg=1.0,
    include_ms=True,
    ms_prune=True,
)

# 2) Load known parameters from JSON (values only)
#    Use apply_vary as you prefer:
#    - "from_json": if your JSON contains "vary" flags you want to honor
#    - "keep_current": ignore "vary" flags in JSON
seed_from_json(
    model,
    filename="/gpfs/Scratch/ufondup/Ti_sample/Ti_fit_checkpoint1.json",
    strict=False,
    show_after=True,
    apply_vary="from_json",    # <-- do not change any vary flags based on JSON
    stage_name=None
)

# 3) Freeze ALL parameters so nothing can change
P = model.pars_mgr.pars
for par in P.__dict__.values():
    if hasattr(par, "vary"):
        par.vary = False

# 4) Prime init values (keeps values identical to JSON)
model._prime_init_values()

# 5) Evaluate once to produce model & metrics (no optimization because nvarys=0)
#    Any method is fine; with nvarys=0 feffit just computes the model and stats.
model.out = model._feffit_with_method(max_nfev=1, method="leastsq")

# 6) Inspect results (will reflect the exact JSON parameters)
print(model.report())                # larch feffit report
model.print_shell_summary()          # your existing shell summary
# Optionally dump a cache for plotting later:
# model.save_plot_cache("eval_bundle.npz")

In [ ]:
model.plot_k()

In [ ]:
model.plot_r()